# Data matching procedure

Combine census data with bank records to produce `data/processed/eda_bank_level.csv` (matched bank records) and `data/processed/eda_area_level.csv` (LSOAs and Scottish Data Zones).


## 1. Locate source files


In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

# Resolve every path from the package root.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
EW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "census_england_wales"
SCOT_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "census_scotland"
BANK_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "bank"
LOOKUP_DATA_DIR = PROJECT_ROOT / "data" / "reference" / "postcode_lookup"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    # England and Wales Census 2021
    "ew_age": EW_DATA_DIR / "census2021-ts007-lsoa-age 5 years.csv",
    "ew_deprivation": EW_DATA_DIR / "census2021-ts011-lsoa-depreviation.csv",
    "ew_health": EW_DATA_DIR / "census2021-ts037-lsoa-General Health.csv",
    "ew_disability": EW_DATA_DIR / "census2021-ts038-lsoa-disability.csv",
    "ew_car": EW_DATA_DIR / "census2021-ts045-lsoa-car.csv",
    "ew_industry": EW_DATA_DIR / "census2021-ts060-lsoa-industry.xlsx",
    "ew_nssec": EW_DATA_DIR / "census2021-ts062-lsoa-NS-SEC.csv",
    "ew_economic_activity": EW_DATA_DIR / "census2021-ts066-lsoa-economic activity.csv",
    "ew_education": EW_DATA_DIR / "census2021-ts067-lsoa-education.csv",

    # Scotland Census 2022
    "s_age": SCOT_DATA_DIR / "Data zone age S.xlsx",
    "s_car": SCOT_DATA_DIR / "Data zone Car S.xlsx",
    "s_deprivation": SCOT_DATA_DIR / "Data zone depreviation S.xlsx",
    "s_disability": SCOT_DATA_DIR / "Data zone disability S.xlsx",
    "s_economic_activity": SCOT_DATA_DIR / "Data zone economic activity S.xlsx",
    "s_education": SCOT_DATA_DIR / "Data zone education S.xlsx",
    "s_health": SCOT_DATA_DIR / "Data zone General Health S.xlsx",
    "s_industry": SCOT_DATA_DIR / "Data zone Industry S.xlsx",
    "s_nssec": SCOT_DATA_DIR / "Data zone NS-SEC S.xlsx",

    # Bank records and postcode-to-LSOA lookup
    "bank": BANK_DATA_DIR / "geolytix_uk_open_bank_branches.csv",
    "postcode_lookup": LOOKUP_DATA_DIR / "Prepared pcd to lsoa.csv",
}

missing = [name for name, path in FILES.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"These source files were not found: {missing}")

display(pd.DataFrame({
    "key": FILES.keys(),
    "path": [str(p) for p in FILES.values()],
    "size_mb": [round(p.stat().st_size / 1024**2, 3) for p in FILES.values()],
}))


,key,path,size_mb
0,ew_age,C:\Users\liwen\Documents\ERP Topological data ...,11.018
1,ew_deprivation,C:\Users\liwen\Documents\ERP Topological data ...,1.978
2,ew_health,C:\Users\liwen\Documents\ERP Topological data ...,1.871
3,ew_disability,C:\Users\liwen\Documents\ERP Topological data ...,2.105
4,ew_car,C:\Users\liwen\Documents\ERP Topological data ...,1.716
5,ew_industry,C:\Users\liwen\Documents\ERP Topological data ...,2.506
6,ew_nssec,C:\Users\liwen\Documents\ERP Topological data ...,2.373
7,ew_economic_activity,C:\Users\liwen\Documents\ERP Topological data ...,4.272
8,ew_education,C:\Users\liwen\Documents\ERP Topological data ...,2.132
9,s_age,C:\Users\liwen\Documents\ERP Topological data ...,0.371


## 2. Data loading

Reshape census tables, retain valid Scottish Data Zone codes, and validate data merges.


In [2]:
def merge_one_to_one(left, right, label):
    """Merge on area_code with one-to-one validation and a stable row count."""
    if right["area_code"].duplicated().any():
        dupes = right.loc[right["area_code"].duplicated(), "area_code"].head().tolist()
        raise ValueError(f"{label}: duplicate area_code values found: {dupes}")
    before = len(left)
    merged = left.merge(right, on="area_code", how="left", validate="one_to_one")
    if len(merged) != before:
        raise ValueError(f"{label}: row count changed during merge")
    return merged


def read_ew_long(path, category_col, value_col="Observation"):
    """Reshape England and Wales census data to one row per LSOA."""
    raw = pd.read_csv(path, low_memory=False)
    code_col = "Lower layer Super Output Areas Code"
    name_col = "Lower layer Super Output Areas"
    raw[value_col] = pd.to_numeric(raw[value_col], errors="coerce")
    wide = raw.pivot(
        index=[code_col, name_col],
        columns=category_col,
        values=value_col,
    ).reset_index()
    wide.columns.name = None
    return wide.rename(columns={code_col: "area_code", name_col: "area_name"})


def read_ew_wide(path):
    """Load wide census data and standardise area column names."""
    raw = pd.read_csv(path, low_memory=False)
    return raw.rename(columns={"geography code": "area_code", "geography": "area_name"})


def read_scot_simple(path):
    """Load Scottish census data, retaining only valid Data Zone codes."""
    raw = pd.read_excel(path, sheet_name="Data Sheet 0")
    code_col = raw.columns[0]
    valid_code = raw[code_col].astype("string").str.fullmatch(r"S\d+")
    return raw.loc[valid_code.fillna(False)].copy()


def normalise_postcode(series):
    """Convert postcodes to uppercase and remove whitespace."""
    return (
        series.astype("string")
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
        .str.strip()
    )


## 3. Prepare England and Wales census data


In [3]:
def build_england_wales_census():
    # Six age groups align with the Scottish categories.
    age = read_ew_long(FILES["ew_age"], "Age (6 categories)").rename(
        columns={
            "Aged 15 years and under": "age_0_15",
            "Aged 16 to 24 years": "age_16_24",
            "Aged 25 to 34 years": "age_25_34",
            "Aged 35 to 49 years": "age_35_49",
            "Aged 50 to 64 years": "age_50_64",
            "Aged 65 years and over": "age_65_plus",
        }
    )
    age_cols = ["age_0_15", "age_16_24", "age_25_34", "age_35_49", "age_50_64", "age_65_plus"]
    age["population_total"] = age[age_cols].sum(axis=1)
    census = age[["area_code", "area_name", "population_total", *age_cols]].copy()

    # Combine deprivation dimensions into 0, 1 and 2+ to align with Scotland.
    dep_raw = read_ew_wide(FILES["ew_deprivation"])
    dep = pd.DataFrame({
        "area_code": dep_raw["area_code"],
        "households_total": dep_raw["Household deprivation: Total: All households; measures: Value"],
        "deprived_0": dep_raw["Household deprivation: Household is not deprived in any dimension; measures: Value"],
        "deprived_1": dep_raw["Household deprivation: Household is deprived in one dimension; measures: Value"],
        "deprived_2_plus": dep_raw[[
            "Household deprivation: Household is deprived in two dimensions; measures: Value",
            "Household deprivation: Household is deprived in three dimensions; measures: Value",
            "Household deprivation: Household is deprived in four dimensions; measures: Value",
        ]].sum(axis=1),
    })
    census = merge_one_to_one(census, dep, "EW deprivation")

    health_raw = read_ew_wide(FILES["ew_health"])
    health = health_raw[[
        "area_code",
        "General health: Total: All usual residents",
        "General health: Very good health",
        "General health: Good health",
        "General health: Fair health",
        "General health: Bad health",
        "General health: Very bad health",
    ]].rename(columns={
        "General health: Total: All usual residents": "health_total",
        "General health: Very good health": "health_very_good",
        "General health: Good health": "health_good",
        "General health: Fair health": "health_fair",
        "General health: Bad health": "health_bad",
        "General health: Very bad health": "health_very_bad",
    })
    census = merge_one_to_one(census, health, "EW health")

    disability_raw = read_ew_wide(FILES["ew_disability"])
    disability = disability_raw[[
        "area_code",
        "Disability: Total: All usual residents",
        "Disability: Disabled under the Equality Act",
        "Disability: Not disabled under the Equality Act",
    ]].rename(columns={
        "Disability: Total: All usual residents": "disability_total",
        "Disability: Disabled under the Equality Act": "disabled",
        "Disability: Not disabled under the Equality Act": "not_disabled",
    })
    census = merge_one_to_one(census, disability, "EW disability")

    car_raw = read_ew_wide(FILES["ew_car"])
    car = car_raw[[
        "area_code",
        "Number of cars or vans: Total: All households",
        "Number of cars or vans: No cars or vans in household",
        "Number of cars or vans: 1 car or van in household",
        "Number of cars or vans: 2 cars or vans in household",
        "Number of cars or vans: 3 or more cars or vans in household",
    ]].rename(columns={
        "Number of cars or vans: Total: All households": "car_households_total",
        "Number of cars or vans: No cars or vans in household": "cars_0",
        "Number of cars or vans: 1 car or van in household": "cars_1",
        "Number of cars or vans: 2 cars or vans in household": "cars_2",
        "Number of cars or vans: 3 or more cars or vans in household": "cars_3_plus",
    })
    census = merge_one_to_one(census, car, "EW car")

    econ_raw = read_ew_wide(FILES["ew_economic_activity"])
    econ = pd.DataFrame({"area_code": econ_raw["area_code"]})
    econ["economic_activity_total_16_plus"] = econ_raw["Economic activity status: Total: All usual residents aged 16 years and over"]
    econ["employee_part_time"] = (
        econ_raw["Economic activity status: Economically active (excluding full-time students): In employment: Employee: Part-time"]
        + econ_raw["Economic activity status: Economically active and a full-time student: In employment: Employee: Part-time"]
    )
    econ["employee_full_time"] = (
        econ_raw["Economic activity status: Economically active (excluding full-time students): In employment: Employee: Full-time"]
        + econ_raw["Economic activity status: Economically active and a full-time student: In employment: Employee: Full-time"]
    )
    econ["self_employed_part_time"] = econ_raw[[
        "Economic activity status: Economically active (excluding full-time students): In employment: Self-employed with employees: Part-time",
        "Economic activity status: Economically active (excluding full-time students): In employment: Self-employed without employees: Part-time",
        "Economic activity status: Economically active and a full-time student: In employment: Self-employed with employees: Part-time",
        "Economic activity status: Economically active and a full-time student: In employment: Self-employed without employees: Part-time",
    ]].sum(axis=1)
    econ["self_employed_full_time"] = econ_raw[[
        "Economic activity status: Economically active (excluding full-time students): In employment: Self-employed with employees: Full-time",
        "Economic activity status: Economically active (excluding full-time students): In employment: Self-employed without employees: Full-time",
        "Economic activity status: Economically active and a full-time student: In employment: Self-employed with employees: Full-time",
        "Economic activity status: Economically active and a full-time student: In employment: Self-employed without employees: Full-time",
    ]].sum(axis=1)
    econ["unemployed"] = (
        econ_raw["Economic activity status: Economically active (excluding full-time students): Unemployed"]
        + econ_raw["Economic activity status: Economically active and a full-time student: Unemployed"]
    )
    econ["inactive_retired"] = econ_raw["Economic activity status: Economically inactive: Retired"]
    econ["inactive_student"] = econ_raw["Economic activity status: Economically inactive: Student"]
    econ["inactive_home_family"] = econ_raw["Economic activity status: Economically inactive: Looking after home or family"]
    econ["inactive_long_term_sick_disabled"] = econ_raw["Economic activity status: Economically inactive: Long-term sick or disabled"]
    econ["inactive_other"] = econ_raw["Economic activity status: Economically inactive: Other"]
    census = merge_one_to_one(census, econ, "EW economic activity")

    # Qualification groups follow Rudkin et al. (2024), Table 1:
    # Q1 = none + L1; Q2 = L2 + apprenticeship; Q3 = L3 + L4.
    edu_raw = read_ew_wide(FILES["ew_education"])
    edu = pd.DataFrame({"area_code": edu_raw["area_code"]})
    edu["education_total_16_plus"] = edu_raw["Highest level of qualification: Total: All usual residents aged 16 years and over"]
    edu["education_no_qualifications"] = edu_raw["Highest level of qualification: No qualifications"]
    edu["education_apprenticeship"] = edu_raw["Highest level of qualification: Apprenticeship"]
    edu["education_low"] = edu_raw[[
        "Highest level of qualification: No qualifications",
        "Highest level of qualification: Level 1 and entry level qualifications",
    ]].sum(axis=1)
    edu["education_middle"] = edu_raw[[
        "Highest level of qualification: Level 2 qualifications",
        "Highest level of qualification: Apprenticeship",
    ]].sum(axis=1)
    edu["education_higher"] = edu_raw[[
        "Highest level of qualification: Level 3 qualifications",
        "Highest level of qualification: Level 4 qualifications and above",
    ]].sum(axis=1)
    edu["education_other_qualifications"] = edu_raw[
        "Highest level of qualification: Other qualifications"
    ]
    edu_reconciliation = edu[[
        "education_low", "education_middle", "education_higher",
        "education_other_qualifications",
    ]].sum(axis=1)
    if not edu_reconciliation.eq(edu["education_total_16_plus"]).all():
        raise ValueError("England/Wales education categories do not reconcile to the age-16-plus total.")
    census = merge_one_to_one(census, edu, "EW education")

    ns_raw = read_ew_wide(FILES["ew_nssec"])
    ns = pd.DataFrame({"area_code": ns_raw["area_code"]})
    ns["nssec_total_16_plus"] = ns_raw["National Statistics Socio-economic Classification (NS-SEC): Total: All usual residents aged 16 years and over"]
    ns["nssec_managerial_professional"] = ns_raw[[
        "National Statistics Socio-economic Classification (NS-SEC): L1, L2 and L3 Higher managerial, administrative and professional occupations",
        "National Statistics Socio-economic Classification (NS-SEC): L4, L5 and L6 Lower managerial, administrative and professional occupations",
    ]].sum(axis=1)
    ns["nssec_intermediate"] = ns_raw["National Statistics Socio-economic Classification (NS-SEC): L7 Intermediate occupations"]
    ns["nssec_small_employers_own_account"] = ns_raw["National Statistics Socio-economic Classification (NS-SEC): L8 and L9 Small employers and own account workers"]
    ns["nssec_lower_supervisory_technical"] = ns_raw["National Statistics Socio-economic Classification (NS-SEC): L10 and L11 Lower supervisory and technical occupations"]
    ns["nssec_semi_routine_routine"] = ns_raw[[
        "National Statistics Socio-economic Classification (NS-SEC): L12 Semi-routine occupations",
        "National Statistics Socio-economic Classification (NS-SEC): L13 Routine occupations",
    ]].sum(axis=1)
    ns["nssec_never_worked_long_term_unemployed"] = ns_raw["National Statistics Socio-economic Classification (NS-SEC): L14.1 and L14.2 Never worked and long-term unemployed"]
    ns["nssec_full_time_students"] = ns_raw["National Statistics Socio-economic Classification (NS-SEC): L15 Full-time students"]
    census = merge_one_to_one(census, ns, "EW NS-SEC")

    ind_raw = pd.read_excel(FILES["ew_industry"], sheet_name="census2021_ts060_lsoa_wide")
    ind = ind_raw.rename(columns={
        "lsoa_code": "area_code",
        "cat_agriculture_energy_water": "industry_agriculture_energy_water",
        "cat_manufacturing": "industry_manufacturing",
        "cat_construction": "industry_construction",
        "cat_distribution_hotels_restaurants": "industry_distribution_hotels_restaurants",
        "cat_transport_communication": "industry_transport_communication",
        "cat_financial_realestate_professional": "industry_finance_real_estate_professional_admin",
        "cat_public_admin_education_health": "industry_public_admin_education_health",
        "cat_other": "industry_other",
        "total_employed": "industry_total_with_code",
    })
    industry_cols = [
        "industry_agriculture_energy_water",
        "industry_manufacturing",
        "industry_construction",
        "industry_distribution_hotels_restaurants",
        "industry_transport_communication",
        "industry_finance_real_estate_professional_admin",
        "industry_public_admin_education_health",
        "industry_other",
    ]
    ind = ind[["area_code", "industry_total_with_code", *industry_cols]]
    census = merge_one_to_one(census, ind, "EW industry")

    census.insert(2, "country", census["area_code"].str[0].map({"E": "England", "W": "Wales"}))
    census.insert(3, "geography_type", "LSOA")
    census.insert(4, "census_year", 2021)
    return census


england_wales_census = build_england_wales_census()
print(england_wales_census.shape)
display(england_wales_census.head())


(35672, 65)


,area_code,area_name,country,geography_type,census_year,population_total,age_0_15,age_16_24,age_25_34,age_35_49,age_50_64,age_65_plus,households_total,deprived_0,deprived_1,deprived_2_plus,health_total,health_very_good,health_good,health_fair,health_bad,health_very_bad,disability_total,disabled,not_disabled,car_households_total,cars_0,cars_1,cars_2,cars_3_plus,economic_activity_total_16_plus,employee_part_time,employee_full_time,self_employed_part_time,self_employed_full_time,unemployed,inactive_retired,inactive_student,inactive_home_family,inactive_long_term_sick_disabled,inactive_other,education_total_16_plus,education_no_qualifications,education_apprenticeship,education_low,education_middle,education_higher,education_other_qualifications,nssec_total_16_plus,nssec_managerial_professional,nssec_intermediate,nssec_small_employers_own_account,nssec_lower_supervisory_technical,nssec_semi_routine_routine,nssec_never_worked_long_term_unemployed,nssec_full_time_students,industry_total_with_code,industry_agriculture_energy_water,industry_manufacturing,industry_construction,industry_distribution_hotels_restaurants,industry_transport_communication,industry_finance_real_estate_professional_admin,industry_public_admin_education_health,industry_other
0,E01000001,City of London 001A,England,LSOA,2021,1477,125,110,291,336,245,370,838,548,253,37,1475,859,468,119,18,11,1475,152,1323,837,555,243,29,10,1351,79,577,85,124,38,305,76,33,9,25,1353,32,12,59,65,1213,16,1354,987,71,115,11,49,37,84,865,2,25,10,52,127,398,168,83
1,E01000002,City of London 001B,England,LSOA,2021,1384,86,123,300,301,297,277,824,542,253,29,1384,836,423,93,24,8,1385,147,1238,824,578,208,26,12,1298,65,597,67,150,33,233,85,27,0,41,1298,23,4,51,56,1173,18,1299,989,55,95,7,25,40,88,880,1,22,10,50,110,474,142,71
2,E01000003,City of London 001C,England,LSOA,2021,1614,110,114,353,368,384,285,1014,487,387,140,1613,790,587,185,44,7,1612,220,1392,1017,826,169,15,7,1509,119,678,92,111,70,243,65,40,43,48,1505,132,12,193,102,1175,35,1506,933,133,128,32,140,72,68,998,1,12,25,99,133,434,209,85
3,E01000005,City of London 001E,England,LSOA,2021,1098,139,195,216,208,235,105,479,163,177,139,1100,501,388,132,63,16,1099,189,910,479,375,92,9,3,960,108,335,25,30,75,96,131,40,52,68,961,197,32,261,152,518,30,962,284,60,57,42,205,151,163,498,1,3,14,85,49,177,138,31
4,E01000006,Barking and Dagenham 016A,England,LSOA,2021,1844,414,222,305,471,276,156,554,204,241,109,1847,880,679,225,53,10,1847,208,1639,554,183,251,95,25,1428,220,423,101,140,65,117,106,131,75,50,1433,295,39,453,209,705,66,1431,330,128,237,70,325,185,156,885,3,26,128,194,107,176,230,21


## 4. Prepare Scotland census data


In [4]:
def build_scotland_census():
    age_raw = read_scot_simple(FILES["s_age"])
    age = age_raw.rename(columns={
        "Data Zone (2022)": "area_code",
        "Aged 0 to 15": "age_0_15",
        "Aged 16 to 24": "age_16_24",
        "Aged 25 to 34": "age_25_34",
        "Aged 35 to 49": "age_35_49",
        "Aged 50 to 64": "age_50_64",
        "Aged 65 and over": "age_65_plus",
        "Total": "population_total",
    })
    age["area_name"] = pd.NA
    census = age[[
        "area_code", "area_name", "population_total",
        "age_0_15", "age_16_24", "age_25_34",
        "age_35_49", "age_50_64", "age_65_plus",
    ]].copy()

    dep_raw = read_scot_simple(FILES["s_deprivation"])
    dep = dep_raw.rename(columns={
        "Data Zone (2022)": "area_code",
        "Household not deprived in any dimension": "deprived_0",
        "Household deprived in 1 dimension": "deprived_1",
        "Household deprived in 2 dimension": "deprived_2_plus",
        "Total": "households_total",
    })[["area_code", "households_total", "deprived_0", "deprived_1", "deprived_2_plus"]]
    census = merge_one_to_one(census, dep, "Scotland deprivation")

    health = read_scot_simple(FILES["s_health"]).rename(columns={
        "Data Zone (2022)": "area_code",
        "Very good": "health_very_good",
        "Good": "health_good",
        "Fair": "health_fair",
        "Bad": "health_bad",
        "Very bad": "health_very_bad",
        "Total": "health_total",
    })[[
        "area_code", "health_total", "health_very_good",
        "health_good", "health_fair", "health_bad", "health_very_bad",
    ]]
    census = merge_one_to_one(census, health, "Scotland health")

    disability = read_scot_simple(FILES["s_disability"]).rename(columns={
        "Data Zone (2022)": "area_code",
        "Yes, limited a lot or a little": "disabled",
        "No": "not_disabled",
        "Total": "disability_total",
    })[["area_code", "disability_total", "disabled", "not_disabled"]]
    census = merge_one_to_one(census, disability, "Scotland disability")

    car_raw = read_scot_simple(FILES["s_car"])
    car = pd.DataFrame({
        "area_code": car_raw["Data Zone (2022)"],
        "car_households_total": car_raw["Total"],
        "cars_0": car_raw["Number of cars or vans in household: No cars or vans"],
        "cars_1": car_raw["Number of cars or vans in household: One car or van"],
        "cars_2": car_raw["Number of cars or vans in household: Two cars or vans"],
        "cars_3_plus": car_raw[[
            "Number of cars or vans in household: Three cars or vans",
            "Number of cars or vans in household: Four or more cars or vans",
        ]].sum(axis=1),
    })
    census = merge_one_to_one(census, car, "Scotland car")

    econ_raw = read_scot_simple(FILES["s_economic_activity"])
    under_16 = "Not applicable (aged less than 16)\xa0"
    econ = pd.DataFrame({
        "area_code": econ_raw["Data Zone (2022)"],
        "economic_activity_total_16_plus": econ_raw["Total"] - econ_raw[under_16],
        "employee_part_time": econ_raw["Economically active - Employee - Part-time"],
        "employee_full_time": econ_raw["Economically active - Employee - Full-time"],
        "self_employed_part_time": econ_raw["Economically active - Self-employed - Part-time"],
        "self_employed_full_time": econ_raw["Economically active - Self-employed - Full-time"],
        "unemployed": econ_raw["Economically active - Unemployed"],
        "inactive_retired": econ_raw["Economically inactive - Retired"],
        "inactive_student": econ_raw["Economically inactive - Student"],
        "inactive_home_family": econ_raw["Economically inactive - Looking after home or family"],
        "inactive_long_term_sick_disabled": econ_raw["Economically inactive - Long-term sick or disabled"],
        "inactive_other": econ_raw["Economically inactive - Other"],
    })
    census = merge_one_to_one(census, econ, "Scotland economic activity")

    
    edu_raw = read_scot_simple(FILES["s_education"])
    edu = pd.DataFrame({
        "area_code": edu_raw["Data Zone (2022)"],
        "education_total_16_plus": edu_raw["Total"] - edu_raw[under_16],
        "education_no_qualifications": edu_raw["No qualifications"],
        "education_apprenticeship": edu_raw["Apprenticeship qualifications"],
    })
    edu["education_low"] = edu_raw[[
        "No qualifications",
        "Lower school qualifications",
    ]].sum(axis=1)
    edu["education_middle"] = edu_raw[[
        "Upper school qualifications",
        "Apprenticeship qualifications",
    ]].sum(axis=1)
    edu["education_higher"] = edu_raw[[
        "Further Education and sub-degree Higher Education qualifications incl. HNC/HNDs",
        "Degree level qualifications or above Education qualifications not already mentioned (including foreign qualifications)",
    ]].sum(axis=1)
    edu["education_other_qualifications"] = pd.NA
    edu_group_sum = edu[["education_low", "education_middle", "education_higher"]].sum(axis=1)
    edu_gap = edu["education_total_16_plus"] - edu_group_sum
    if edu_gap.isna().any() or edu_gap.abs().max() > 20:
        raise ValueError("Scotland education groups fail the published-total reconciliation check.")
    print("Scotland education reconciliation gap (published total minus grouped categories):")
    display(edu_gap.describe().to_frame(name="count_difference").T)
    census = merge_one_to_one(census, edu, "Scotland education")

    ns_raw = read_scot_simple(FILES["s_nssec"])
    ns = pd.DataFrame({
        "area_code": ns_raw["Data Zone (2022)"],
        "nssec_total_16_plus": ns_raw["Total"] - ns_raw[under_16],
        "nssec_managerial_professional": ns_raw[[
            "Higher managerial, administrative and professional occupations",
            "Lower managerial, administrative and professional occupations",
        ]].sum(axis=1),
        "nssec_intermediate": ns_raw["Intermediate occupations"],
        "nssec_small_employers_own_account": ns_raw["Small employers and own account workers"],
        "nssec_lower_supervisory_technical": ns_raw["Lower supervisory and technical occupations"],
        "nssec_semi_routine_routine": ns_raw[["Semi-routine occupations", "Routine occupations"]].sum(axis=1),
        "nssec_never_worked_long_term_unemployed": ns_raw["Never worked and long-term unemployed"],
        "nssec_full_time_students": ns_raw["Full-time students"],
    })
    census = merge_one_to_one(census, ns, "Scotland NS-SEC")

    ind_raw = read_scot_simple(FILES["s_industry"])
    ind = ind_raw.rename(columns={
        "Data Zone (2022)": "area_code",
        "A, B, D, E. Agriculture, energy and water": "industry_agriculture_energy_water",
        "C. Manufacturing": "industry_manufacturing",
        "F. Construction": "industry_construction",
        "G, I. Distribution, hotels and restaurants": "industry_distribution_hotels_restaurants",
        "H, J. Transport and communication": "industry_transport_communication",
        "K, L, M, N. Financial, real estate, professional and administrative activities": "industry_finance_real_estate_professional_admin",
        "O, P, Q. Public administration, education and health": "industry_public_admin_education_health",
        "R, S, T, U. Other": "industry_other",
    })
    industry_cols = [
        "industry_agriculture_energy_water",
        "industry_manufacturing",
        "industry_construction",
        "industry_distribution_hotels_restaurants",
        "industry_transport_communication",
        "industry_finance_real_estate_professional_admin",
        "industry_public_admin_education_health",
        "industry_other",
    ]
    ind["industry_total_with_code"] = ind[industry_cols].sum(axis=1)
    ind = ind[["area_code", "industry_total_with_code", *industry_cols]]
    census = merge_one_to_one(census, ind, "Scotland industry")

    census.insert(2, "country", "Scotland")
    census.insert(3, "geography_type", "Data Zone")
    census.insert(4, "census_year", 2022)
    return census


scotland_census = build_scotland_census()
print(scotland_census.shape)
display(scotland_census.head())


Scotland education reconciliation gap (published total minus grouped categories):


,count,mean,std,min,25%,50%,75%,max
count_difference,7392.0,0.053707,3.082935,-15.0,-2.0,0.0,2.0,16.0


(7392, 65)


,area_code,area_name,country,geography_type,census_year,population_total,age_0_15,age_16_24,age_25_34,age_35_49,age_50_64,age_65_plus,households_total,deprived_0,deprived_1,deprived_2_plus,health_total,health_very_good,health_good,health_fair,health_bad,health_very_bad,disability_total,disabled,not_disabled,car_households_total,cars_0,cars_1,cars_2,cars_3_plus,economic_activity_total_16_plus,employee_part_time,employee_full_time,self_employed_part_time,self_employed_full_time,unemployed,inactive_retired,inactive_student,inactive_home_family,inactive_long_term_sick_disabled,inactive_other,education_total_16_plus,education_no_qualifications,education_apprenticeship,education_low,education_middle,education_higher,education_other_qualifications,nssec_total_16_plus,nssec_managerial_professional,nssec_intermediate,nssec_small_employers_own_account,nssec_lower_supervisory_technical,nssec_semi_routine_routine,nssec_never_worked_long_term_unemployed,nssec_full_time_students,industry_total_with_code,industry_agriculture_energy_water,industry_manufacturing,industry_construction,industry_distribution_hotels_restaurants,industry_transport_communication,industry_finance_real_estate_professional_admin,industry_public_admin_education_health,industry_other
0,S01013482,<NA>,Scotland,Data Zone,2022,969.0,147.0,83.0,112.0,203.0,219.0,205.0,491.0,270.0,151.0,70.0,969.0,487.0,321.0,121.0,33.0,9.0,969.0,203.0,766.0,491.0,79.0,246.0,130.0,36.0,822.0,104.0,304.0,23.0,54.0,18.0,209.0,20.0,35.0,25.0,24.0,822.0,84.0,65.0,220.0,145.0,457.0,<NA>,822.0,335.0,95.0,75.0,68.0,147.0,60.0,46.0,759.0,99.0,57.0,38.0,112.0,45.0,110.0,242.0,56.0
1,S01013483,<NA>,Scotland,Data Zone,2022,758.0,125.0,36.0,81.0,130.0,157.0,226.0,366.0,157.0,143.0,66.0,758.0,364.0,236.0,134.0,19.0,7.0,758.0,174.0,584.0,366.0,79.0,165.0,90.0,32.0,633.0,84.0,221.0,12.0,15.0,15.0,238.0,10.0,12.0,15.0,9.0,633.0,118.0,45.0,242.0,99.0,292.0,<NA>,633.0,242.0,72.0,43.0,46.0,167.0,40.0,24.0,597.0,60.0,38.0,40.0,101.0,37.0,92.0,191.0,38.0
2,S01013484,<NA>,Scotland,Data Zone,2022,542.0,92.0,44.0,88.0,134.0,106.0,81.0,292.0,172.0,91.0,29.0,542.0,289.0,163.0,70.0,14.0,5.0,542.0,91.0,448.0,292.0,43.0,166.0,68.0,15.0,450.0,61.0,221.0,12.0,24.0,13.0,77.0,14.0,15.0,12.0,4.0,450.0,37.0,28.0,125.0,70.0,254.0,<NA>,450.0,184.0,55.0,36.0,39.0,85.0,27.0,27.0,438.0,55.0,22.0,22.0,87.0,32.0,65.0,129.0,26.0
3,S01013485,<NA>,Scotland,Data Zone,2022,560.0,108.0,45.0,57.0,112.0,110.0,127.0,270.0,135.0,87.0,47.0,560.0,277.0,177.0,81.0,20.0,6.0,560.0,121.0,439.0,270.0,54.0,132.0,65.0,15.0,452.0,69.0,168.0,9.0,21.0,8.0,119.0,16.0,20.0,17.0,6.0,452.0,67.0,55.0,147.0,95.0,210.0,<NA>,452.0,164.0,53.0,38.0,35.0,109.0,19.0,34.0,433.0,33.0,30.0,24.0,81.0,36.0,56.0,144.0,29.0
4,S01013486,<NA>,Scotland,Data Zone,2022,621.0,95.0,49.0,54.0,117.0,149.0,153.0,297.0,153.0,90.0,54.0,621.0,269.0,223.0,100.0,25.0,4.0,621.0,160.0,461.0,297.0,70.0,125.0,79.0,19.0,526.0,66.0,180.0,10.0,25.0,11.0,145.0,28.0,17.0,27.0,8.0,526.0,74.0,56.0,184.0,111.0,224.0,<NA>,526.0,178.0,58.0,42.0,45.0,118.0,51.0,34.0,460.0,46.0,30.0,33.0,77.0,20.0,61.0,153.0,40.0


## 5. Combine census data


In [5]:
census_master = pd.concat([england_wales_census, scotland_census], ignore_index=True)
assert census_master["area_code"].is_unique

print("Census master:", census_master.shape)
display(census_master.groupby(["country", "geography_type", "census_year"]).size())


Census master: (43064, 65)


country   geography_type  census_year
England   LSOA            2021           33755
Scotland  Data Zone       2022            7392
Wales     LSOA            2021            1917
dtype: int64

## 6. Clean bank records and match area codes


In [6]:
def prepare_bank_data():
    banks = pd.read_csv(FILES["bank"], low_memory=False)

    # Preserve the original closure year for traceability.
    banks["close_year_raw"] = banks["close_year"]

    # Extract years from eight-digit dates; flag and clear invalid years.
    eight_digit_date = banks["close_year_raw"].between(10_000_000, 99_999_999)
    banks.loc[eight_digit_date, "close_year"] = banks.loc[eight_digit_date, "close_year_raw"] // 10_000
    banks["close_year"] = pd.to_numeric(banks["close_year"], errors="coerce")
    invalid_year = banks["close_year"].notna() & ~banks["close_year"].between(1900, 2030)
    banks["close_year_format_issue"] = invalid_year
    banks.loc[invalid_year, "close_year"] = pd.NA

    lookup = pd.read_csv(FILES["postcode_lookup"], usecols=["pcds", "lsoa21cd"], low_memory=False)
    banks["postcode_key"] = normalise_postcode(banks["postcode"])
    lookup["postcode_key"] = normalise_postcode(lookup["pcds"])
    lookup = lookup.drop_duplicates("postcode_key")

    banks = banks.merge(
        lookup[["postcode_key", "lsoa21cd"]],
        on="postcode_key",
        how="left",
        validate="many_to_one",
    ).rename(columns={"lsoa21cd": "area_code"})

    # Exclude Northern Ireland, the Channel Islands and the Isle of Man.
    excluded_code = banks["area_code"].astype("string").str.startswith(("N", "L", "M")).fillna(False)
    excluded_region = banks["region"].eq("Northern Ireland")
    banks = banks.loc[~(excluded_code | excluded_region)].copy()

    country_from_code = banks["area_code"].astype("string").str[0].map({"E": "England", "W": "Wales", "S": "Scotland"})
    country_from_region = banks["region"].where(banks["region"].isin(["Scotland", "Wales"]), "England")
    banks["country"] = country_from_code.fillna(country_from_region)
    return banks


banks = prepare_bank_data()
print("Bank rows after GB filtering:", len(banks))
print("Matched area_code:", banks["area_code"].notna().sum())
print("Unmatched area_code:", banks["area_code"].isna().sum())
display(banks["status"].value_counts(dropna=False))
display(banks.loc[banks["close_year_format_issue"], ["id", "branch_name", "close_year_raw", "close_year"]])


Bank rows after GB filtering: 11744
Matched area_code: 11564
Unmatched area_code: 180


status
Closed     6486
Open       5248
Closing      10
Name: count, dtype: int64

,id,branch_name,close_year_raw,close_year
848,8548,Santander Didsbury,7.0,NaN
9204,11118,Barclays Local Helston,6.0,NaN


## 7. Create the bank-level dataset


In [7]:
bank_eda = banks.merge(
    census_master,
    on="area_code",
    how="left",
    suffixes=("_bank", "_census"),
    validate="many_to_one",
)

assert bank_eda["id"].is_unique
print("bank_eda:", bank_eda.shape)
print("Bank rows with Census match:", bank_eda["population_total"].notna().sum())
display(bank_eda.head())


bank_eda: (11744, 87)
Bank rows with Census match: 11564


,id,brand_full,brand_short,branch_name,branch_type,add_one,add_two,suburb,town,region,postcode,long_wgs84,lat_wgs84,status,close_month,close_year,open_year,po_dist,close_year_raw,close_year_format_issue,postcode_key,area_code,country_bank,area_name,country_census,geography_type,census_year,population_total,age_0_15,age_16_24,age_25_34,age_35_49,age_50_64,age_65_plus,households_total,deprived_0,deprived_1,deprived_2_plus,health_total,health_very_good,health_good,health_fair,health_bad,health_very_bad,disability_total,disabled,not_disabled,car_households_total,cars_0,cars_1,cars_2,cars_3_plus,economic_activity_total_16_plus,employee_part_time,employee_full_time,self_employed_part_time,self_employed_full_time,unemployed,inactive_retired,inactive_student,inactive_home_family,inactive_long_term_sick_disabled,inactive_other,education_total_16_plus,education_no_qualifications,education_apprenticeship,education_low,education_middle,education_higher,education_other_qualifications,nssec_total_16_plus,nssec_managerial_professional,nssec_intermediate,nssec_small_employers_own_account,nssec_lower_supervisory_technical,nssec_semi_routine_routine,nssec_never_worked_long_term_unemployed,nssec_full_time_students,industry_total_with_code,industry_agriculture_energy_water,industry_manufacturing,industry_construction,industry_distribution_hotels_restaurants,industry_transport_communication,industry_finance_real_estate_professional_admin,industry_public_admin_education_health,industry_other
0,9381,TSB,TSB,TSB Hamilton,Branch,20 Quarry Place Shopping Arcad,NaN,Central Hamilton,Hamilton,Scotland,ML3 7BB,-4.035629,55.775277,Open,NaN,NaN,NaN,897,NaN,False,ML37BB,S01020062,Scotland,<NA>,Scotland,Data Zone,2022.0,1087.0,104.0,111.0,129.0,174.0,206.0,366.0,659.0,257.0,213.0,192.0,1087.0,420.0,327.0,221.0,83.0,39.0,1087.0,354.0,733.0,659.0,302.0,265.0,77.0,13.0,983.0,119.0,302.0,15.0,39.0,27.0,352.0,23.0,25.0,56.0,25.0,983.0,247.0,74.0,383.0,173.0,426.0,<NA>,983.0,318.0,133.0,64.0,72.0,239.0,82.0,72.0,892.0,23.0,91.0,55.0,193.0,58.0,152.0,273.0,47.0
1,4128,Lloyds,LL,Lloyds Accrington,Branch,2 Peel Street,NaN,Hillock Vale,Accrington,North West,BB5 1EP,-2.363137,53.754043,Open,NaN,NaN,NaN,46,NaN,False,BB51EP,E01025036,England,Hyndburn 008B,England,LSOA,2021.0,1717.0,326.0,195.0,328.0,335.0,309.0,224.0,897.0,312.0,307.0,278.0,1718.0,668.0,578.0,311.0,124.0,37.0,1718.0,440.0,1278.0,897.0,435.0,346.0,93.0,23.0,1393.0,148.0,471.0,35.0,54.0,86.0,203.0,70.0,87.0,159.0,80.0,1392.0,351.0,83.0,496.0,319.0,533.0,44,1390.0,255.0,132.0,125.0,75.0,501.0,217.0,85.0,707.0,11.0,107.0,51.0,184.0,66.0,79.0,195.0,14.0
2,5712,Nationwide,NAI,Nationwide Hamilton,Branch,57-59 Quarry Street,NaN,Central Hamilton,Hamilton,Scotland,ML3 7AH,-4.034835,55.775039,Open,NaN,NaN,NaN,953,NaN,False,ML37AH,S01020062,Scotland,<NA>,Scotland,Data Zone,2022.0,1087.0,104.0,111.0,129.0,174.0,206.0,366.0,659.0,257.0,213.0,192.0,1087.0,420.0,327.0,221.0,83.0,39.0,1087.0,354.0,733.0,659.0,302.0,265.0,77.0,13.0,983.0,119.0,302.0,15.0,39.0,27.0,352.0,23.0,25.0,56.0,25.0,983.0,247.0,74.0,383.0,173.0,426.0,<NA>,983.0,318.0,133.0,64.0,72.0,239.0,82.0,72.0,892.0,23.0,91.0,55.0,193.0,58.0,152.0,273.0,47.0
3,403,Barclays,BB,Barclays Local Rawtenstall,Branch,36,Bank Street,NaN,Rawtenstall,North West,BB4 7QW,-2.284940,53.702596,Closed,6.0,2022.0,NaN,145,2022.0,False,BB47QW,E01025373,England,Rossendale 004C,England,LSOA,2021.0,1648.0,307.0,138.0,193.0,306.0,346.0,358.0,725.0,432.0,207.0,86.0,1646.0,901.0,532.0,153.0,49.0,11.0,1648.0,239.0,1409.0,723.0,105.0,282.0,263.0,73.0,1339.0,180.0,484.0,78.0,68.0,22.0,341.0,51.0,49.0,34.0,32.0,1341.0,180.0,67.0,290.0,255.0,776.0,20,1339.0,564.0,164.0,179.0,68.0,199.0,101.0,64.0,810.0,14.0,72.0,63.0,170.0,53.0,120.0,282.0,36.0
4,1808,The Co-operative Bank,CO,Co-op Bank Burnley,Branch,60 St James St,NaN,Central Burnley,Burnley,North West,BB11 1NH,-2.243382,53.789092,Closed,6.0,2016.0,NaN,161,2016.0,False,BB111NH,E01024877,England,Burnley 003D,England,LSOA,2021.0,2054.0,500.0,

## 8. Create the area-level dataset


In [8]:
matched_banks = banks.loc[banks["area_code"].isin(census_master["area_code"])].copy()

bank_counts = (
    matched_banks.groupby(["area_code", "status"], observed=True)
    .size()
    .unstack(fill_value=0)
    .rename(columns={"Open": "bank_count_open", "Closed": "bank_count_closed", "Closing": "bank_count_closing"})
    .reset_index()
)

for column in ["bank_count_open", "bank_count_closed", "bank_count_closing"]:
    if column not in bank_counts.columns:
        bank_counts[column] = 0

bank_counts["bank_count_total"] = bank_counts[["bank_count_open", "bank_count_closed", "bank_count_closing"]].sum(axis=1)

area_eda = census_master.merge(
    bank_counts[["area_code", "bank_count_total", "bank_count_open", "bank_count_closed", "bank_count_closing"]],
    on="area_code",
    how="left",
    validate="one_to_one",
)

bank_count_columns = ["bank_count_total", "bank_count_open", "bank_count_closed", "bank_count_closing"]
area_eda[bank_count_columns] = area_eda[bank_count_columns].fillna(0).astype("int64")

assert area_eda["area_code"].is_unique
print("area_eda:", area_eda.shape)
display(area_eda.head())


area_eda: (43064, 69)


,area_code,area_name,country,geography_type,census_year,population_total,age_0_15,age_16_24,age_25_34,age_35_49,age_50_64,age_65_plus,households_total,deprived_0,deprived_1,deprived_2_plus,health_total,health_very_good,health_good,health_fair,health_bad,health_very_bad,disability_total,disabled,not_disabled,car_households_total,cars_0,cars_1,cars_2,cars_3_plus,economic_activity_total_16_plus,employee_part_time,employee_full_time,self_employed_part_time,self_employed_full_time,unemployed,inactive_retired,inactive_student,inactive_home_family,inactive_long_term_sick_disabled,inactive_other,education_total_16_plus,education_no_qualifications,education_apprenticeship,education_low,education_middle,education_higher,education_other_qualifications,nssec_total_16_plus,nssec_managerial_professional,nssec_intermediate,nssec_small_employers_own_account,nssec_lower_supervisory_technical,nssec_semi_routine_routine,nssec_never_worked_long_term_unemployed,nssec_full_time_students,industry_total_with_code,industry_agriculture_energy_water,industry_manufacturing,industry_construction,industry_distribution_hotels_restaurants,industry_transport_communication,industry_finance_real_estate_professional_admin,industry_public_admin_education_health,industry_other,bank_count_total,bank_count_open,bank_count_closed,bank_count_closing
0,E01000001,City of London 001A,England,LSOA,2021,1477.0,125.0,110.0,291.0,336.0,245.0,370.0,838.0,548.0,253.0,37.0,1475.0,859.0,468.0,119.0,18.0,11.0,1475.0,152.0,1323.0,837.0,555.0,243.0,29.0,10.0,1351.0,79.0,577.0,85.0,124.0,38.0,305.0,76.0,33.0,9.0,25.0,1353.0,32.0,12.0,59.0,65.0,1213.0,16,1354.0,987.0,71.0,115.0,11.0,49.0,37.0,84.0,865.0,2.0,25.0,10.0,52.0,127.0,398.0,168.0,83.0,1,0,1,0
1,E01000002,City of London 001B,England,LSOA,2021,1384.0,86.0,123.0,300.0,301.0,297.0,277.0,824.0,542.0,253.0,29.0,1384.0,836.0,423.0,93.0,24.0,8.0,1385.0,147.0,1238.0,824.0,578.0,208.0,26.0,12.0,1298.0,65.0,597.0,67.0,150.0,33.0,233.0,85.0,27.0,0.0,41.0,1298.0,23.0,4.0,51.0,56.0,1173.0,18,1299.0,989.0,55.0,95.0,7.0,25.0,40.0,88.0,880.0,1.0,22.0,10.0,50.0,110.0,474.0,142.0,71.0,2,2,0,0
2,E01000003,City of London 001C,England,LSOA,2021,1614.0,110.0,114.0,353.0,368.0,384.0,285.0,1014.0,487.0,387.0,140.0,1613.0,790.0,587.0,185.0,44.0,7.0,1612.0,220.0,1392.0,1017.0,826.0,169.0,15.0,7.0,1509.0,119.0,678.0,92.0,111.0,70.0,243.0,65.0,40.0,43.0,48.0,1505.0,132.0,12.0,193.0,102.0,1175.0,35,1506.0,933.0,133.0,128.0,32.0,140.0,72.0,68.0,998.0,1.0,12.0,25.0,99.0,133.0,434.0,209.0,85.0,0,0,0,0
3,E01000005,City of London 001E,England,LSOA,2021,1098.0,139.0,195.0,216.0,208.0,235.0,105.0,479.0,163.0,177.0,139.0,1100.0,501.0,388.0,132.0,63.0,16.0,1099.0,189.0,910.0,479.0,375.0,92.0,9.0,3.0,960.0,108.0,335.0,25.0,30.0,75.0,96.0,131.0,40.0,52.0,68.0,961.0,197.0,32.0,261.0,152.0,518.0,30,962.0,284.0,60.0,57.0,42.0,205.0,151.0,163.0,498.0,1.0,3.0,14.0,85.0,49.0,177.0,138.0,31.0,1,0,1,0
4,E01000006,Barking and Dagenham 016A,England,LSOA,2021,1844.0,414.0,222.0,305.0,471.0,276.0,156.0,554.0,204.0,241.0,109.0,1847.0,880.0,679.0,225.0,53.0,10.0,1847.0,208.0,1639.0,554.0,183.0,251.0,95.0,25.0,1428.0,220.0,423.0,101.0,140.0,65.0,117.0,106.0,131.0,75.0,50.0,1433.0,295.0,39.0,453.0,209.0,705.0,66,1431.0,330.0,128.0,237.0,70.0,325.0,185.0,156.0,885.0,3.0,26.0,128.0,194.0,107.0,176.0,230.0,21.0,0,0,0,0


## 9. Check data quality


In [9]:
quality_report = {
    "bank_rows": len(bank_eda),
    "bank_unique_ids": bank_eda["id"].nunique(),
    "bank_rows_with_area_code": int(bank_eda["area_code"].notna().sum()),
    "bank_rows_with_census_match": int(bank_eda["population_total"].notna().sum()),
    "area_rows": len(area_eda),
    "area_unique_codes": area_eda["area_code"].nunique(),
    "areas_with_any_bank_record": int((area_eda["bank_count_total"] > 0).sum()),
}
display(pd.Series(quality_report, name="value"))

country_summary = area_eda.groupby("country").agg(
    area_rows=("area_code", "size"),
    bank_records=("bank_count_total", "sum"),
    open_records=("bank_count_open", "sum"),
    closed_records=("bank_count_closed", "sum"),
    closing_records=("bank_count_closing", "sum"),
)
display(country_summary)

assert quality_report["area_rows"] == quality_report["area_unique_codes"]
assert (area_eda[bank_count_columns] >= 0).all().all()
assert (area_eda["bank_count_total"] == area_eda[["bank_count_open", "bank_count_closed", "bank_count_closing"]].sum(axis=1)).all()


bank_rows                      11744
bank_unique_ids                11744
bank_rows_with_area_code       11564
bank_rows_with_census_match    11564
area_rows                      43064
area_unique_codes              43064
areas_with_any_bank_record      4100
Name: value, dtype: int64

,area_rows,bank_records,open_records,closed_records,closing_records
country,,,,,
England,33755,9214,4029,5177,8
Scotland,7392,1560,807,752,1
Wales,1917,790,407,382,1


## 10. Save the analytical datasets

Save both CSV files to `data/processed`, retaining only census-matched records in the bank-level dataset.


In [10]:
OUTPUT_DIR.mkdir(exist_ok=True)

bank_output = OUTPUT_DIR / "eda_bank_level.csv"
area_output = OUTPUT_DIR / "eda_area_level.csv"

# Retain only bank records matched to census data.
bank_final = bank_eda.loc[bank_eda["population_total"].notna()].copy()
excluded_bank_rows = len(bank_eda) - len(bank_final)

bank_final.to_csv(bank_output, index=False, encoding="utf-8-sig")
area_eda.to_csv(area_output, index=False, encoding="utf-8-sig")

print("Saved:")
print(bank_output)
print(area_output)
print(f"Excluded bank rows without census data: {excluded_bank_rows}")


Saved:
C:\Users\liwen\Documents\ERP Topological data science\材料提交\Additional_Materials_Submission\data\processed\eda_bank_level.csv
C:\Users\liwen\Documents\ERP Topological data science\材料提交\Additional_Materials_Submission\data\processed\eda_area_level.csv
Excluded bank rows without census data: 180
